In [35]:
    #inference_batch_size=24
from BiLSTMLanguageModeler import BiLSTMLanguageModel
feature_learning_path = "../model/Sen_old_model/pretrain_models/feature_learning_model/bilstm_256-04.hdf5"
fz_model = BiLSTMLanguageModel(
    seq_len=23,
    vocab_size=27,
    embedding_dim=20,
    hidden_dim=256,
    n_hidden=2,
    n_epochs=24,
    batch_size=8,
    inference_batch_size=8,
    cache_dir='',
    seed=41,
    verbose=True,
)
fz_model.model_.summary()
fz_model.model_.load_weights('../model/Sen_old_model/pretrain_models/feature_learning_model/bilstm_256-04.hdf5')
    

Model: "model_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, 22)]         0                                            
__________________________________________________________________________________________________
input_8 (InputLayer)            [(None, 22)]         0                                            
__________________________________________________________________________________________________
embedding_3 (Embedding)         (None, 22, 20)       560         input_7[0][0]                    
                                                                 input_8[0][0]                    
__________________________________________________________________________________________________
lstm_6 (LSTM)                   (None, 22, 256)      283648      embedding_3[0][0]          

In [37]:
import os

In [36]:
import tensorflow.keras.models as models

disc_model = models.load_model('../model/Sen_old_model/pretrain_models/discriminator-model/integrated_model')

In [8]:
disc_model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
batch_normalization (BatchNo (None, 512)               2048      
_________________________________________________________________
dense_4 (Dense)              (None, 1)                 513       
Total params: 2,561
Trainable params: 1,537
Non-trainable params: 1,024
_________________________________________________________________


In [38]:
import pandas as pd 
import numpy as np 
def getVocabDictionary():
    AAs = [
    'A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H',
    'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W',
    'Y', 'V', 'X', 'Z', 'J', 'U', 'B',
    ]
    vocab = {
                amino_acid: index+1  for index, amino_acid in enumerate(sorted(AAs))
            }
            
    return vocab
def featurize_seqs(seqs):
        vocabulary = getVocabDictionary()
        start_int = len(vocabulary) + 1
        end_int = len(vocabulary) + 2
        sorted_seqs = sorted(seqs)
        X = np.concatenate([
            np.array([ start_int ] + [
                vocabulary[word] for word in seq
            ] + [ end_int ]) for seq in sorted_seqs
        ]).reshape(-1, 1)
        lens = np.array([ len(seq) + 2 for seq in sorted_seqs ])
        assert(sum(lens) == X.shape[0])
        return X, lens
def read_combined_window_file(file_path):
    combined_features = []
    df = pd.read_csv(file_path)
    #Appending features | 
    for row in df.itertuples():
        combined_features.append(row.wild)
        combined_features.append(row.mutated)
    print("Combined Features Length: ", len(combined_features))
    #Ensure that features lenght is always even
    assert(len(combined_features) % 2 == 0)
    return combined_features

def get_features(file_path):
    seqs = read_combined_window_file(file_path)
    X_cat, lengths = featurize_seqs(seqs)
    y_embed_output = fz_model.transform(X_cat, lengths )
    print("Shape of output previously: ", y_embed_output.shape) #  (rows, 22, 512)
    #Reducing the feature taking the avarage from middle axis
    y_embed_output = np.average(y_embed_output, axis=1)
    print("Reduced feature Shape after average: ", y_embed_output.shape)
    return y_embed_output

def compute_save_pred_score(model_base_path, save_file_name, sig_path, non_sig_path):
    '''
    Computes Prediction scores for given dataset eg: validation , greaney, Baum
    and saves the correspodning results along with target values.
    -------
    Parameters:
    model_base_path: Base Directory where the execution result is saved.
    save_file_name : Name of the CSV file that will be saved
    {val_preds | baum_preds | greaney_preds}.csv
    
    sig_path: Significant file path 
    non_sig_path : Nonsignificant file path
    '''
   
    non_sig_data  = get_features(non_sig_path)
    sig_data =  get_features(sig_path)

    sig_len =  len(sig_data)
    sig_features_output = np.ones( (sig_len, 1) )
    non_sig_len = len(non_sig_data)
    nonsig_features_output = np.zeros( (non_sig_len, 1) )

    features = np.concatenate( (sig_data, non_sig_data), axis=0)
    targets = np.concatenate( (sig_features_output, nonsig_features_output), axis=0 )

    print("Evaluating  dataset for : ")
    disc_model.evaluate(features, targets)

    #Analyze  predictions | plot AUC_ROC / Confusion Matrix / Precision Recall
    y_preds = disc_model.predict(features)
    y_preds = y_preds.flatten() #Connvert 2D data into 1 D [[1.0], [0.97]] -> [1.0, 0.97]
    
    targets = targets.flatten()
    
    
    #Saving records to output file
    df_new = pd.DataFrame({'target': targets, 'predicted': y_preds})
    save_path = model_base_path + os.path.sep + save_file_name
    df_new.to_csv(save_path, index=None)
    print(f'File Saved Successfully to path: ',save_path)
    
    

    return targets, y_preds


In [ ]:
# targets, y_preds = analyze_validation_dataset()
model_base_path = '../model/Sen_old_model'
data_file = {
    'VAL': {
        'SAVE_FILE_NAME': 'val_preds.csv',
        'SIG_PATH': '../data/disc/train/sig_combined_windows_test.csv',
        'NONSIG_PATH' : '../data/disc/train/non_sig_combined_windows_test.csv'
    },
    'GREANEY': {
        'SAVE_FILE_NAME': 'greany_preds.csv',
        'SIG_PATH': '../data/test/greany_sig_combined_windowed_seqs.csv',
        'NONSIG_PATH' : '../data/test/greany_non_sig_combined_windowed_seqs.csv'
    },
    'BAUM': {
        'SAVE_FILE_NAME': 'baum_preds.csv',
        'SIG_PATH': '../data/test/baum_sig_combined_windowed_seqs.csv',
        'NONSIG_PATH' : '../data/test/baum_non_sig_combined_windowed_seqs.csv'
    }
}

for key, record  in data_file.items():
        file_name = record['SAVE_FILE_NAME']
        sig_path = record['SIG_PATH']
        non_sig_path = record['NONSIG_PATH']
        print(file_name , sig_path, non_sig_path)
        compute_save_pred_score(model_base_path, file_name, sig_path, non_sig_path)




val_preds.csv ../data/disc/train/sig_combined_windows_test.csv ../data/disc/train/non_sig_combined_windows_test.csv
Combined Features Length:  72756
2025-02-27 10:44:37.014340 | Embedding...
200079/200079 [==============================] - 2357s 12ms/step
2025-02-27 11:23:55.368106 | Done embedding.
Shape of output previously:  (72756, 22, 512)
Reduced feature Shape after average:  (72756, 512)
Combined Features Length:  16000
2025-02-27 11:23:58.681587 | Embedding...
44000/44000 [==============================] - 516s 12ms/step
2025-02-27 11:32:34.742720 | Done embedding.
Shape of output previously:  (16000, 22, 512)
Reduced feature Shape after average:  (16000, 512)
Evaluating  dataset for : 
2774/2774 [==============================] - 10s 3ms/step - loss: 0.3364 - accuracy: 0.9171 - auc: 0.9017
File Saved Successfully to path:  ../model/Sen_old_model/val_preds.csv
greaney_preds.csv ../data/test/greany_sig_combined_windowed_seqs.csv ../data/test/greany_non_sig_combined_windowed_seqs